In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)
pd.set_option('display.precision', 3)
pd.set_option('mode.chained_assignment', None)

!rm -rf /kaggle/working/*

In [2]:
!nvidia-smi

Thu Mar 26 17:21:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
train=pd.read_csv("/kaggle/input/competitions/ai-will-detect-smokers/train.csv")

test=pd.read_csv("/kaggle/input/competitions/ai-will-detect-smokers/test.csv")

sub=pd.read_csv("/kaggle/input/competitions/ai-will-detect-smokers/sample_submission.csv")

print(f"Train Data Shape: {train.shape}")
print(f"Check out Null Values: {train.isnull().sum()}")
print(f"Train Data info: {train.info()}")

print("#"*100)
print("#"*100)
print("#"*100)
print("#"*100)
print(f"Test Data Shape: {test.shape}")
print(f"Check out Null Values: {test.isnull().sum()}")
print(f"Test Data info: {test.info()}")

Train Data Shape: (15000, 24)
Check out Null Values: id                     0
age                    0
height(cm)             0
weight(kg)             0
waist(cm)              0
eyesight(left)         0
eyesight(right)        0
hearing(left)          0
hearing(right)         0
systolic               0
relaxation             0
fasting blood sugar    0
Cholesterol            0
triglyceride           0
HDL                    0
LDL                    0
hemoglobin             0
Urine protein          0
serum creatinine       0
AST                    0
ALT                    0
Gtp                    0
dental caries          0
smoking                0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   15000 non-null  int64  
 1   age                  15000 non-null  float64
 2   height(cm)          

In [4]:
train.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,fasting blood sugar,Cholesterol,triglyceride,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,smoking
0,0,30.0,180.0,70.0,84.0,1.2,1.2,1.0,1.0,120.0,84.0,95.0,209.0,91.0,61.0,129.0,15.8,1.0,1.2,16.0,11.0,16.0,0.0,0.0
1,1,30.0,175.0,95.0,107.4,1.2,1.5,1.0,1.0,138.0,88.0,82.0,251.0,102.0,80.0,151.0,17.7,1.0,0.9,50.0,106.0,52.0,1.0,1.0
2,2,45.0,155.0,60.0,81.5,1.0,1.0,1.0,1.0,114.0,67.0,86.0,198.0,46.0,71.0,118.0,12.9,1.0,0.8,13.0,11.0,17.0,0.0,0.0
3,3,60.0,170.0,65.0,89.0,0.8,0.8,1.0,1.0,101.0,62.0,111.0,187.0,91.0,55.0,114.0,14.6,1.0,1.1,20.0,14.0,37.0,0.0,1.0
4,4,40.0,160.0,55.0,78.0,1.2,1.2,1.0,1.0,123.0,78.0,107.0,174.0,70.0,58.0,102.0,16.4,1.0,1.1,19.0,21.0,16.0,0.0,0.0


In [5]:
test.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,relaxation,fasting blood sugar,Cholesterol,triglyceride,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries
0,15000,45.0,165.0,75.0,91.0,1.2,1.0,1.0,1.0,122.0,72.0,96.0,189.0,99.0,48.0,121.0,15.4,1.0,1.0,32.0,34.0,23.0,0.0
1,15001,40.0,160.0,60.0,82.0,0.9,0.9,1.0,1.0,119.0,67.0,96.0,200.0,64.0,68.0,119.0,13.5,1.0,0.6,18.0,17.0,20.0,0.0
2,15002,25.0,170.0,70.0,79.0,1.5,1.5,1.0,1.0,124.0,78.0,89.0,173.0,70.0,72.0,87.0,15.2,1.0,0.9,17.0,22.0,23.0,0.0
3,15003,40.0,160.0,80.0,94.0,0.9,1.0,1.0,1.0,120.0,80.0,101.0,177.0,101.0,54.0,103.0,14.0,1.0,0.5,35.0,30.0,13.0,0.0
4,15004,60.0,165.0,70.0,95.0,1.0,1.2,1.0,1.0,117.0,66.0,126.0,187.0,140.0,56.0,104.0,14.1,1.0,0.9,15.0,17.0,27.0,0.0


#  Feature Engineering

In [6]:
def make_features(df):
    df = df.copy()
    
    # BMI
    df['bmi'] = df['weight(kg)'] / (df['height(cm)'] / 100) ** 2
    
    # Liver enzyme ratios (smokers have higher ALT/AST)
    df['ast_alt_ratio'] = df['AST'] / (df['ALT'] + 1)
    df['gtp_alt_ratio']  = df['Gtp'] / (df['ALT'] + 1)
    
    # Cholesterol ratios
    df['hdl_ldl_ratio']  = df['HDL'] / (df['LDL'] + 1)
    df['chol_hdl_ratio'] = df['Cholesterol'] / (df['HDL'] + 1)
    
    # Blood pressure pulse pressure
    df['pulse_pressure'] = df['systolic'] - df['relaxation']
    
    # Log transforms (skewed features)
    for col in ['triglyceride', 'Gtp', 'ALT', 'AST', 'serum creatinine']:
        df[f'log_{col}'] = np.log1p(df[col])
    
    # Waist-to-height ratio
    df['waist_height_ratio'] = df['waist(cm)'] / df['height(cm)']
    
    return df

train = make_features(train)
test  = make_features(test)

FEATURES = [c for c in train.columns if c not in ['id', 'smoking']]
X = train[FEATURES]
y = train['smoking']
X_test = test[FEATURES]
print(f"Total features: {len(FEATURES)}")

Total features: 34


# XGBoost

In [7]:
xgb_params = {
    'n_estimators': 2000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'use_label_encoder': False,
    'eval_metric': 'auc',
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',      
    'tree_method': 'hist', 
    'early_stopping_rounds': 200
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_oof  = np.zeros(len(train))
xgb_pred = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)
    
    xgb_oof[val_idx]  = model.predict_proba(X_val)[:, 1]
    xgb_pred          += model.predict_proba(X_test)[:, 1] / 5
    score = roc_auc_score(y_val, xgb_oof[val_idx])
    print(f"Fold {fold+1} XGB AUC: {score:.5f}")

print(f"\nXGB OOF AUC: {roc_auc_score(y, xgb_oof):.5f}")

Fold 1 XGB AUC: 0.89633
Fold 2 XGB AUC: 0.88937
Fold 3 XGB AUC: 0.89391
Fold 4 XGB AUC: 0.88476
Fold 5 XGB AUC: 0.88186

XGB OOF AUC: 0.88920


# LightGBM

In [8]:
lgb_params = {
    'n_estimators': 2000,
    'num_leaves': 63,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 20,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
}

lgb_oof  = np.zeros(len(train))
lgb_pred = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)])
    
    lgb_oof[val_idx]  = model.predict_proba(X_val)[:, 1]
    lgb_pred          += model.predict_proba(X_test)[:, 1] / 5
    score = roc_auc_score(y_val, lgb_oof[val_idx])
    print(f"Fold {fold+1} LGB AUC: {score:.5f}")

print(f"\nLGB OOF AUC: {roc_auc_score(y, lgb_oof):.5f}")

Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[92]	valid_0's binary_logloss: 0.397348
Fold 1 LGB AUC: 0.89067
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[90]	valid_0's binary_logloss: 0.398622
Fold 2 LGB AUC: 0.88697
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[126]	valid_0's binary_logloss: 0.393797
Fold 3 LGB AUC: 0.89067
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[121]	valid_0's binary_logloss: 0.402988
Fold 4 LGB AUC: 0.88449
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[84]	valid_0's binary_logloss: 0.409341
Fold 5 LGB AUC: 0.87971

LGB OOF AUC: 0.88635


# CatBoost

In [9]:

cat_oof  = np.zeros(len(train))
cat_pred = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = CatBoostClassifier(
        iterations=2000, depth=6,
        learning_rate=0.05,
        eval_metric='AUC',
        early_stopping_rounds=200,
        random_state=42, verbose=0,
        task_type='GPU',
        devices='0'
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    
    cat_oof[val_idx]  = model.predict_proba(X_val)[:, 1]
    cat_pred += model.predict_proba(X_test)[:, 1] / 5
    score = roc_auc_score(y_val, cat_oof[val_idx])
    print(f"Fold {fold+1} CAT AUC: {score:.5f}")

print(f"\nCAT OOF AUC: {roc_auc_score(y, cat_oof):.5f}")

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1 CAT AUC: 0.89508


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2 CAT AUC: 0.88512


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 3 CAT AUC: 0.89169


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4 CAT AUC: 0.88606


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5 CAT AUC: 0.88201

CAT OOF AUC: 0.88790


# Random Forest 

In [10]:
rf_oof  = np.zeros(len(train))
rf_pred = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model =  RandomForestClassifier(
        n_estimators=500,
        max_depth=16,
        max_features=0.8,
        random_state=42,
       
    )
    model.fit(X_tr.values.astype('float32'), y_tr.values.astype('float32'))

    rf_oof[val_idx]  = model.predict_proba(X_val.values.astype('float32'))[:, 1]
    rf_pred          += model.predict_proba(X_test.values.astype('float32'))[:, 1] / 5
    score = roc_auc_score(y_val, rf_oof[val_idx])
    print(f"Fold {fold+1} RF AUC: {score:.5f}")

print(f"\nRF OOF AUC: {roc_auc_score(y, rf_oof):.5f}")

Fold 1 RF AUC: 0.88278
Fold 2 RF AUC: 0.87987
Fold 3 RF AUC: 0.88284
Fold 4 RF AUC: 0.87394
Fold 5 RF AUC: 0.87405

RF OOF AUC: 0.87852


# Extra Trees

In [11]:
et_oof  = np.zeros(len(train))
et_pred = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )
    model.fit(X_tr, y_tr)

    et_oof[val_idx]  = model.predict_proba(X_val)[:, 1]
    et_pred          += model.predict_proba(X_test)[:, 1] / 5
    score = roc_auc_score(y_val, et_oof[val_idx])
    print(f"Fold {fold+1} ET AUC: {score:.5f}")

print(f"\nET OOF AUC: {roc_auc_score(y, et_oof):.5f}")

Fold 1 ET AUC: 0.88719
Fold 2 ET AUC: 0.88093
Fold 3 ET AUC: 0.88890
Fold 4 ET AUC: 0.87708
Fold 5 ET AUC: 0.87387

ET OOF AUC: 0.88150


# Neural Network with PyTorch GPU

In [12]:
from sklearn.neural_network import MLPClassifier
from tqdm.notebook import tqdm
from tabulate import tabulate

nn_oof  = np.zeros(len(train))
nn_pred = np.zeros(len(test))
fold_rows = []

scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

for fold, (tr_idx, val_idx) in enumerate(tqdm(skf.split(X_scaled, y), total=5, desc="Folds")):
    X_tr,  X_val = X_scaled[tr_idx],  X_scaled[val_idx]
    y_tr,  y_val = y.values[tr_idx],  y.values[val_idx]

    model = MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation='relu',
        batch_size=512,
        learning_rate_init=1e-3,
        max_iter=100,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42,
    )
    model.fit(X_tr, y_tr)

    val_preds        = model.predict_proba(X_val)[:, 1]
    nn_oof[val_idx]  = val_preds
    nn_pred         += model.predict_proba(X_test_scaled)[:, 1] / 5
    score            = roc_auc_score(y_val, val_preds)
    fold_rows.append([f"Fold {fold+1}", f"{score:.5f}"])

fold_rows.append(["OOF AUC", f"{roc_auc_score(y, nn_oof):.5f}"])
print(tabulate(fold_rows, headers=["Fold", "AUC"], tablefmt="rounded_outline"))

Folds:   0%|          | 0/5 [00:00<?, ?it/s]

╭─────────┬─────────╮
│ Fold    │     AUC │
├─────────┼─────────┤
│ Fold 1  │ 0.89215 │
│ Fold 2  │ 0.88223 │
│ Fold 3  │ 0.88644 │
│ Fold 4  │ 0.88296 │
│ Fold 5  │ 0.87707 │
│ OOF AUC │ 0.88347 │
╰─────────┴─────────╯


# Logistic Regression 

In [13]:
lr_oof  = np.zeros(len(train))
lr_pred = np.zeros(len(test))

scaler2      = StandardScaler()
X_sc         = scaler2.fit_transform(X)
X_test_sc    = scaler2.transform(X_test)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_sc, y)):
    model = LogisticRegression(C=0.1, max_iter=1000, random_state=42, n_jobs=-1)
    model.fit(X_sc[tr_idx], y.values[tr_idx])

    lr_oof[val_idx] = model.predict_proba(X_sc[val_idx])[:, 1]
    lr_pred         += model.predict_proba(X_test_sc)[:, 1] / 5
    score = roc_auc_score(y.values[val_idx], lr_oof[val_idx])
    print(f"Fold {fold+1} LR  AUC: {score:.5f}")

print(f"\nLR OOF AUC:  {roc_auc_score(y, lr_oof):.5f}")

Fold 1 LR  AUC: 0.87713
Fold 2 LR  AUC: 0.87342
Fold 3 LR  AUC: 0.87618
Fold 4 LR  AUC: 0.87264
Fold 5 LR  AUC: 0.87005

LR OOF AUC:  0.87384


In [14]:
all_oof = np.column_stack([xgb_oof, lgb_oof, cat_oof,rf_oof,  et_oof,  nn_oof, lr_oof])
all_pred = np.column_stack([xgb_pred, lgb_pred, cat_pred,rf_pred,  et_pred,  nn_pred, lr_pred])

model_names = ['XGBoost', 'LightGBM', 'CatBoost',
               'RandForest', 'ExtraTrees', 'NeuralNet', 'LogReg']

print("Individual OOF AUC scores:")
for name, oof in zip(model_names, all_oof.T):
    print(f"  {name:<12}: {roc_auc_score(y, oof):.5f}")

def neg_auc(weights):
    weights = np.array(weights)
    weights = weights / weights.sum()
    blended = all_oof @ weights
    return -roc_auc_score(y, blended)

constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}
bounds      = [(0, 1)] * len(model_names)
init_w      = [1 / len(model_names)] * len(model_names)

result  = minimize(neg_auc, init_w, method='SLSQP',
                   bounds=bounds, constraints=constraints)
best_w  = result.x / result.x.sum()

print("\nOptimized weights:")
for name, w in zip(model_names, best_w):
    print(f"  {name:<12}: {w:.4f}")

Individual OOF AUC scores:
  XGBoost     : 0.88920
  LightGBM    : 0.88635
  CatBoost    : 0.88790
  RandForest  : 0.87852
  ExtraTrees  : 0.88150
  NeuralNet   : 0.88347
  LogReg      : 0.87384

Optimized weights:
  XGBoost     : 0.1429
  LightGBM    : 0.1429
  CatBoost    : 0.1429
  RandForest  : 0.1429
  ExtraTrees  : 0.1429
  NeuralNet   : 0.1429
  LogReg      : 0.1429


In [15]:
final_pred = all_pred @ best_w
print(f"Final Ensemble OOF AUC: {roc_auc_score(y, all_oof @ best_w):.5f}")

sub['smoking'] = final_pred
sub.to_csv('submission.csv', index=False)
print("submission.csv saved!")
sub.head()

Final Ensemble OOF AUC: 0.88889
submission.csv saved!


,id,smoking
0,15000,0.307
1,15001,0.020
2,15002,0.446
3,15003,0.046
4,15004,0.570
